# Governance Memo: Interpretability And Model Risk

This notebook converts the technical outputs from both v0.7 interpretability track notebooks (Alpine Crest Private Bank and NovaBank Digital) into a single stakeholder-readable governance summary. It consumes the model-documentation template and monitoring checklist (slice 3) and synthesises explanations, threshold decisions, false-positive concentration, model documentation, and monitoring needs for both tracks.

Governance language stays educational: this memo is an exercise, not a certification or legal-advice artifact. **A model should not be judged by headline accuracy.**

> Educational exercise on synthetic data. No real Client, account, or transaction data.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from banking_fraud_lab import (
    build_digital_banking_features,
    build_learner_facing_views,
    build_model_documentation,
    build_monitoring_checklist,
    build_private_banking_features,
    concentrate_false_positives,
    evaluate_alert_scores,
    extract_feature_importance,
    generate_digital_fraud_scenarios_world,
    generate_private_banking_transaction_fraud_world,
    recommend_lowest_cost_threshold,
)
from banking_fraud_lab.features import (
    DIGITAL_BANKING_FEATURE_FAMILIES,
    PRIVATE_BANKING_FEATURE_FAMILIES,
)

pd.set_option("display.max_columns", 40)

## Build Both Track Baselines

Reproduce the two fitted baselines (Alpine Crest private-banking and NovaBank Digital) so the memo synthesises real model outputs rather than hand-written numbers. Each track produces a fitted model, an alert-score frame, a threshold recommendation, and false-positive concentration.

In [ ]:
def _numeric_columns(frame: pd.DataFrame, families) -> list[str]:
    output_columns = [
        column
        for spec in families
        for column in spec.output_columns
    ]
    return [
        column
        for column in output_columns
        if column in frame.columns and pd.api.types.is_numeric_dtype(frame[column])
    ]


def _fit_baseline(model_frame: pd.DataFrame, numeric_columns: list[str]) -> Pipeline:
    preprocess = ColumnTransformer(
        [("numeric", StandardScaler(), numeric_columns)],
        remainder="drop",
    )
    model = Pipeline(
        [
            ("preprocess", preprocess),
            (
                "model",
                LogisticRegression(
                    random_state=42,
                    solver="lbfgs",
                    max_iter=1000,
                    class_weight="balanced",
                ),
            ),
        ]
    )
    target = model_frame["confirmed_fraud"].astype(bool)
    model.fit(model_frame[numeric_columns], target)
    return model


def private_banking_track() -> dict:
    tables = generate_private_banking_transaction_fraud_world(
        seed=42, scenario_prevalence=0.2
    )
    learner_tables = build_learner_facing_views(tables)
    feature_frame = build_private_banking_features(learner_tables)
    model_frame = (
        learner_tables["cases"][
            ["case_id", "alert_id", "transaction_id", "banking_relationship_id"]
        ]
        .merge(
            learner_tables["alerts"][["alert_id", "alert_type"]],
            on="alert_id",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            learner_tables["case_outcomes"][["case_id", "confirmed_fraud"]],
            on="case_id",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            feature_frame,
            on=["transaction_id", "banking_relationship_id"],
            how="inner",
            validate="one_to_one",
        )
    )
    numeric_columns = _numeric_columns(model_frame, PRIVATE_BANKING_FEATURE_FAMILIES)
    model = _fit_baseline(model_frame, numeric_columns)
    model_frame = model_frame.assign(
        score=model.predict_proba(model_frame[numeric_columns])[:, 1].round(6)
    )
    return {
        "institution": "Alpine Crest Private Bank",
        "track": "private_banking",
        "model": model,
        "model_frame": model_frame,
        "numeric_columns": numeric_columns,
        "learner_tables": learner_tables,
        "case_outcomes": learner_tables["case_outcomes"],
        "alerts": learner_tables["alerts"],
        "detection_pattern_id": "pb_high_value_movement",
    }


def digital_banking_track() -> dict:
    tables = generate_digital_fraud_scenarios_world(
        seed=42, scale="small", scenario_prevalence=0.5, noisy_outcome_rate=0.3
    )
    learner_tables = build_learner_facing_views(tables)
    feature_frame = build_digital_banking_features(learner_tables)
    model_frame = (
        learner_tables["cases"][["case_id", "alert_id", "transaction_id"]]
        .merge(
            learner_tables["alerts"][["alert_id", "alert_type"]],
            on="alert_id",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            learner_tables["case_outcomes"][["case_id", "confirmed_fraud"]],
            on="case_id",
            how="inner",
            validate="one_to_one",
        )
        .merge(feature_frame, on="transaction_id", how="inner", validate="one_to_one")
    )
    numeric_columns = _numeric_columns(model_frame, DIGITAL_BANKING_FEATURE_FAMILIES)
    model = _fit_baseline(model_frame, numeric_columns)
    model_frame = model_frame.assign(
        score=model.predict_proba(model_frame[numeric_columns])[:, 1].round(6)
    )
    return {
        "institution": "NovaBank Digital",
        "track": "digital_banking",
        "model": model,
        "model_frame": model_frame,
        "numeric_columns": numeric_columns,
        "learner_tables": learner_tables,
        "case_outcomes": learner_tables["case_outcomes"],
        "alerts": learner_tables["alerts"],
        "detection_pattern_id": "digital_scam_to_mule",
    }


tracks = [private_banking_track(), digital_banking_track()]
pd.DataFrame(
    [
        {
            "institution": t["institution"],
            "track": t["track"],
            "cases": len(t["model_frame"]),
            "features": len(t["numeric_columns"]),
        }
        for t in tracks
    ]
)

## Synthesise Threshold Decisions

For each track, run the lowest-cost-threshold recommender across capacity and cost, then collect the recommended threshold, total cost, recall, and capacity note into one summary table the memo can cite.

In [ ]:
threshold_rows = []
for track in tracks:
    alert_scores = pd.DataFrame(
        {"alert_id": track["model_frame"]["alert_id"],
         "score": track["model_frame"]["score"]}
    )
    recommendation = recommend_lowest_cost_threshold(
        cases=track["model_frame"][["case_id", "alert_id"]],
        case_outcomes=track["case_outcomes"],
        alert_scores=alert_scores,
        candidate_thresholds=(0.25, 0.5, 0.75),
        alert_capacities=(5, 10, 25),
        investigation_cost_chf=75.0,
        false_positive_cost_chf=25.0,
        missed_fraud_cost_chf=1_000.0,
    )
    summary = recommendation["recommended_summary"]
    threshold_rows.append(
        {
            "institution": track["institution"],
            "track": track["track"],
            "recommended_threshold": recommendation["recommended_threshold"],
            "recommended_capacity": recommendation["recommended_alert_capacity"],
            "total_cost_chf": summary["total_cost_chf"],
            "recall": summary["recall"],
            "precision": summary["precision"],
            "over_capacity_alerts": summary["over_capacity_alerts"],
        }
    )
threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary

## Synthesise Explanations And False-Positive Concentration

For each track, summarise the top feature-importance driver (the explanation the stakeholder memo cites) and the false-positive concentration by alert_type at the recommended threshold. These feed the memo's explanation and review-burden sections.

In [ ]:
explanations = {}
fp_concentrations = {}
for track in tracks:
    alert_scores = pd.DataFrame(
        {"alert_id": track["model_frame"]["alert_id"],
         "score": track["model_frame"]["score"]}
    )
    importance = extract_feature_importance(
        track["model"],
        track["numeric_columns"],
        feature_frame=track["model_frame"][track["numeric_columns"]],
    )
    top_feature = str(
        importance.sort_values("native_importance", ascending=False).iloc[0]["feature"]
    )
    explanations[track["institution"]] = {
        "top_feature": top_feature,
        "top_importance": float(
            importance.set_index("feature").loc[top_feature, "native_importance"]
        ),
    }
    recommended_threshold = threshold_summary.set_index("institution").loc[
        track["institution"], "recommended_threshold"
    ]
    fp_concentrations[track["institution"]] = concentrate_false_positives(
        cases=track["model_frame"][["case_id", "alert_id"]],
        case_outcomes=track["case_outcomes"],
        alert_scores=alert_scores,
        threshold=recommended_threshold,
        segment_columns=["alert_type"],
        alerts=track["alerts"],
    )

fp_summary = pd.DataFrame(
    [
        {
            "institution": institution,
            "false_positive_segments": len(concentration),
            "max_segment_share": float(concentration["false_positive_share"].max())
            if not concentration.empty
            else 0.0,
        }
        for institution, concentration in fp_concentrations.items()
    ]
)
fp_summary

## Synthesise Documentation And Monitoring

Use the model-documentation template and monitoring checklist (slice 3) to produce, for each track, a documentation artifact and a monitoring checklist. These feed the memo's documentation and monitoring sections.

In [ ]:
documentation_artifacts = {}
monitoring_checklists = {}
for track in tracks:
    documentation_artifacts[track["institution"]] = build_model_documentation(
        track["model"],
        institution=track["institution"],
        detection_pattern_id=track["detection_pattern_id"],
        feature_columns=track["numeric_columns"],
        model_frame=track["model_frame"],
        seed=42,
        cost_parameters={
            "investigation_cost_chf": 75.0,
            "false_positive_cost_chf": 25.0,
            "missed_fraud_cost_chf": 1_000.0,
        },
    )
    report = evaluate_alert_scores(
        cases=track["model_frame"][["case_id", "alert_id"]],
        case_outcomes=track["case_outcomes"],
        alert_scores=pd.DataFrame(
            {"alert_id": track["model_frame"]["alert_id"],
             "score": track["model_frame"]["score"]}
        ),
    )
    monitoring_checklists[track["institution"]] = build_monitoring_checklist(
        institution=track["institution"],
        detection_pattern_id=track["detection_pattern_id"],
        evaluation=report,
    )

documentation_index = pd.DataFrame(
    [
        {
            "institution": institution,
            "sections_filled": len(doc["sections"]),
            "estimator": doc["metadata"]["estimator_summary"],
        }
        for institution, doc in documentation_artifacts.items()
    ]
)
documentation_index

## Governance Memo

The memo below is generated from the synthesised outputs. It is structured for a stakeholder audience: threshold decisions, model documentation status, monitoring needs, and limitations. Governance language is educational — no certification or legal-advice claims.

In [ ]:
def capacity_note(over_capacity: int) -> str:
    if over_capacity == 0:
        return "within current investigation capacity"
    return f"{int(over_capacity)} alerts over current capacity"


def monitoring_summary(checklist: dict) -> str:
    """Summarise monitoring dimensions by their computed status."""
    statuses = {
        dimension_id: dimension["status"]
        for dimension_id, dimension in checklist["dimensions"].items()
    }
    review = sorted(d for d, s in statuses.items() if s == "review")
    ok = sorted(d for d, s in statuses.items() if s == "ok")
    not_applicable = sorted(d for d, s in statuses.items() if s == "not_applicable")
    applicable = len(statuses) - len(not_applicable)
    parts = [f"{len(ok)} of {applicable} applicable dimensions ok"]
    if not_applicable:
        parts.append(f"{len(not_applicable)} not applicable")
    if review:
        parts.append(f"review: {', '.join(review)}")
    return "; ".join(parts) + "."


memo_lines = ["# Governance Memo (Educational Draft)", ""]
for row in threshold_summary.itertuples():
    institution = row.institution
    doc = documentation_artifacts[institution]
    checklist = monitoring_checklists[institution]
    filled_sections = sorted(doc["sections"])
    estimator = doc["metadata"]["estimator_summary"]
    explanation = explanations[institution]
    concentration = fp_concentrations[institution]
    fp_note = (
        f"false positives concentrate across {len(concentration)} segment(s)"
        if not concentration.empty
        else "no false positives clear the recommended threshold"
    )
    memo_lines.extend(
        [
            f"## {institution} - {row.track}",
            f"Recommended threshold: {row.recommended_threshold:.2f}, evaluated at an "
            f"alert capacity of {int(row.recommended_capacity)}. At this threshold the "
            f"expected total cost is CHF {row.total_cost_chf:.2f}, with recall "
            f"{row.recall:.2f} and precision {row.precision:.2f}; the alert volume is "
            f"{capacity_note(row.over_capacity_alerts)}.",
            "",
            f"Top explanation driver: feature '{explanation['top_feature']}' "
            f"(normalised importance {explanation['top_importance']:.2f}). "
            "Explanations are inspection aids, not proof of correctness.",
            "",
            f"Review burden: {fp_note} by alert_type.",
            "",
            f"Model documentation: {len(filled_sections)} sections filled "
            f"({', '.join(filled_sections)}) for estimator {estimator}.",
            "",
            f"Monitoring: {monitoring_summary(checklist)}",
            "",
            "Limitations: results use deterministic synthetic data and generated case "
            "outcomes. Explanations are inspection aids, not proof of correctness. A "
            "model should not be judged by headline accuracy.",
            "",
        ]
    )
memo_text = "\n".join(memo_lines)
print(memo_text)

## Governance-Language Guardrail

Confirm the memo carries no certification or legal-advice claims and keeps the limitation-aware framing visible. The repository stays private; this memo adds no public-release language.

In [ ]:
for forbidden in ("certified", "is certified", "legal advice", "compliance requirement"):
    assert forbidden not in memo_text.lower(), f"memo contains forbidden term: {forbidden!r}"
assert "should not be judged by headline accuracy" in memo_text
print("Governance-language guardrail passed: no certification/legal-advice claims; "
      "limitation-aware framing visible.")